# Milestone 1 — Land the IESO hourly demand data

This notebook holds no logic. It only drives the scripts in `src/ingest/`, so that
a Colab run and a local run produce the same thing from the same code.

**On every new Colab session, run cells 1 through 3 in order.** Data lives in Google
Drive, so nothing is lost when the runtime disconnects.

| What | Where it lives | Survives a disconnect? |
|---|---|---|
| Code | GitHub, cloned into `/content/` each session | No, but re-cloning restores it |
| Raw CSVs / Parquet | Google Drive | Yes |
| `reports/` output | The clone; cell 8 copies it to Drive | Once copied, yes |

## 1. Environment

In [ ]:
!pip install -q polars duckdb

from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code

First run clones the repo. Later runs pull the latest commit.

> After editing anything under `src/` on the GitHub website, re-run this cell to
> pick up the new version.

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/ethantosc/ieso-demand-forecast.git"
REPO_DIR = pathlib.Path("/content/ieso-demand-forecast")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd:", pathlib.Path.cwd())
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

## 3. Point `data/` at Drive

The repo's `data/` directory is replaced with a symlink to Drive. This keeps the
scripts in `src/` unaware that they are running on Colab at all — they still see
`data/raw/demand/`, exactly as they would locally.

Safety valve: if the repo's `data/` already holds real data files, this cell stops
rather than overwriting them.

In [ ]:
import pathlib, shutil

DRIVE_DATA = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data")
for sub in ("raw/demand", "raw/forecasts", "staging", "curated"):
    (DRIVE_DATA / sub).mkdir(parents=True, exist_ok=True)

local = pathlib.Path("data")
if local.is_symlink():
    local.unlink()
elif local.exists():
    payload = [p for p in local.rglob("*") if p.suffix.lower() in {".csv", ".parquet", ".xml"}]
    if payload:
        raise SystemExit(
            f"The repo's data/ holds {len(payload)} data file(s). "
            "Decide what to keep before re-running this cell."
        )
    shutil.rmtree(local)

local.symlink_to(DRIVE_DATA)
print("data ->", local.resolve())
print("raw demand files already held:",
      len(list((DRIVE_DATA / "raw/demand").glob("*.csv"))))

## 4. Download the raw CSVs

Annual files from 2002 to the current year. Years already held are skipped; the
current year is re-downloaded every run because it is still being written.

`manifest.json` records each file's download time, byte count and SHA-256. If a
settled year's contents ever change underneath us, a later `--force` run prints
`[CONTENT CHANGED]` instead of silently replacing it.

> If this cell 404s, the URL pattern is wrong. Fall back to dragging the CSVs you
> already have into `MyDrive/ieso-demand-forecast-data/raw/demand/` and skip this cell.

In [ ]:
!python -m src.ingest.fetch_demand

## 5. Land it as Parquet

Reads the annual CSVs, converts hour-ending 1–24 to hour-beginning, derives UTC and
Toronto local time columns, and writes `data/staging/demand_hourly.parquet`.

Raw files are never modified, so this cell is always safe to re-run.

In [ ]:
!python -m src.ingest.demand

## 6. Validate

**This cell is designed to be able to fail, and a failure here is informative.** It
tests the assumptions the ingest depends on: that the CSV timeline is fixed EST,
that every day is complete, that the series has no gaps or duplicates.

- exit 0 → every must-be-empty check passed
- exit 1 → the Parquet is not trustworthy; do not build features on it

Output is written to `reports/data_quality_auto.md`.

In [ ]:
!python -m src.ingest.check_demand

## 7. Read the results

The whole report prints below. The sections that need human judgement:

- **§2 / §7** — whether the fixed-EST assumption holds. §7 should show exactly two
  dates per year: a 23-hour day in March and a 25-hour day in November.
- **§6** — year-file seams, especially 2002 → 2003. A level shift here means the
  files disagree about definition or units.
- **§12 / §13** — the gap between Market and Ontario demand.

In [ ]:
print(pathlib.Path("reports/data_quality_auto.md").read_text())

## 8. Save a copy to Drive

The clone disappears with the session, so the report gets copied out.

To put it in version control later: download the `.md` from Drive and use
**Add file → Upload files** on GitHub to place it in `reports/`.

In [ ]:
import shutil, pathlib

dest = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data/reports")
dest.mkdir(parents=True, exist_ok=True)
shutil.copy("reports/data_quality_auto.md", dest / "data_quality_auto.md")
print("saved ->", dest / "data_quality_auto.md")